# **NEW SECTION**

# **4. Transformer modeli uchun ma’lumotlarni tayyorlash**

## **4.1. Ma’lumotlar to‘plamini yuklash**

### datasets kutubxonasi yordamida 1.2 qismda tayyorlangan DataFrame’dan Dataset obyekti yarating. Ma’lumotlarni train (2500 ta) va test (500 ta) qismlariga ajrating (seed=123).

## **4.2. Matnni tokenizatsiya qilish**





### distilbert-base-uncased tokenizer’ini yuklang.



### Matnlarni tokenizatsiya qiluvchi, padding va truncation qo‘llaydigan preprocess_function yarating (max_length=256).



###Funksiyani train va test to‘plamlariga map metodi orqali qo‘llang.

In [ ]:
!wget https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/IMDB-Dataset.csv


In [ ]:
import pandas as pd

df = pd.read_csv("IMDB-Dataset.csv.1")
df.head()

In [ ]:
positive_reviews = df[df['sentiment'] == 'positive'].sample(1500, random_state=101)
negative_reviews = df[df['sentiment'] == 'negative'].sample(1500, random_state=101)

df = pd.concat([positive_reviews, negative_reviews]).sample(frac=1, random_state=101).reset_index(drop=True)
df
# random_state=101: Tasodifiy tanlovni har doim bir xil natija berishini ta'minlaydi. Ya'ni,
# kodni necha marta ishga tushirilsa ham, bir xil tasodifiy namuna olinadi.
# frac=1: DataFrame'dagi barcha qatorlarni (100%) tanlaydi, lekin ularni tasodifiy tartibda aralashtirib beradi.

In [ ]:
df.shape

In [ ]:
!pip install --upgrade huggingface_hub
from datasets import Dataset

dataset = Dataset.from_pandas(df)
small_train = dataset.shuffle(seed=123 ).select(range(2500))
small_test = dataset.shuffle(seed=123).select(range(500))

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Define label mappings
label2id = {'negative': 0, 'positive': 1}
id2label = {0: 'negative', 1: 'positive'}

def preprocess_function(examples):
    tokenized_inputs = tokenizer(examples["review"], truncation=True, padding='max_length', max_length=256)
    # Convert string labels to numerical IDs
    tokenized_inputs["labels"] = [label2id[label] for label in examples["sentiment"]]
    return tokenized_inputs

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_test.map(preprocess_function, batched=True)

In [ ]:
print(small_train.column_names)
print(small_test.column_names)

# **5. Transformer modelini o‘qitish va baholash**

## **5.1. Modelni yuklash va sozlash**


### distilbert-base-uncased asosida AutoModelForSequenceClassification modelini yuklang


### TrainingArguments sozlamalarini yarating: num_train_epochs=3, logging_steps=100.


### Trainer obyektini yarating.

## **5.2. Modelni o‘qitish va baholash**


### trainer.train() metodi bilan modelni shug‘ullantiring.


### trainer.evaluate() metodini chaqirib, modelning eval_loss ko‘rsatkichini tahlil qiling.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

In [ ]:
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification
import os
import torch

# Ensure label2id and id2label are defined as in cell YjznBuEQZF38, or reuse them if they are global
# For robustness, we redefine them here for this specific cell's execution context if not already global.
label2id = {'negative': 0, 'positive': 1}
id2label = {0: 'negative', 1: 'positive'}

model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(label2id), # Use the number of defined labels
    id2label=id2label,
    label2id=label2id
)

# Modelni aniq CPU ga o'tkazish (Agar yuqoridagi muhit o'zgaruvchisi yetarli bo'lmasa)
# model.to('cpu') kodini faqat CUDA mavjud bo'lganda ishlatish maqsadga muvofiq.
# Agar CUDA_VISIBLE_DEVICES="" o'rnatilgan bo'lsa, PyTorch CUDA ni ko'rmaydi.
# Lekin agar model hali ham GPU da bo'lsa, uni CPU ga o'tkazish kerak.
if torch.cuda.is_available() and model.device.type == 'cuda':
    model.to('cpu')

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=100,
    report_to='none'
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

trainer.train()

trainer.evaluate()

# **Modelni baholash va sinovdan o‘tkazish**

In [ ]:
texts = [
    "I hated this movie","Bu filmni juda uzoq kutgandim, lekin umuman yoqmadi.",
    "I waited for this film for a long time, but I didn't like it at all.",
    " Aktyorlar jamoasi ajoyib, syujet ham juda qiziqarli ekan.",
    "The cast is amazing, and the plot is very interesting.",
    "Filmning vizual effektlari yaxshi, ammo hikoyasi zaif.",
    "The film's visual effects are good, but the story is weak.",

    "this movie is a masterpiece", "a complete waste of time",
    "This movie was an absolute masterpiece, highly recommend watching it!"
]

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# Inputs ni to'g'ridan-to'g'ri CPU ga yuborish
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=256).to('cpu')

# Modelni ham CPU ga o'tkazilganligiga ishonch hosil qilish
if torch.cuda.is_available() and model.device.type == 'cuda':
    model.to('cpu')

with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    predictions = torch.argmax(probs, dim=1)

label_map = {0: "negative", 1: 'possitive'}

for text, pred, prob in zip(texts, predictions, probs):
    print(text, label_map[pred.item()], prob[pred.item()].item())

# **DEPLOYMENT**

# **Website yaratish va modeldan hayotda foydalanish**

In [ ]:
import torch

def predict_sentiment(text):
  # Inputs ni CPU ga yuborish, chunki model ham CPU da ishlamoqda
  inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=256).to('cpu')

  with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    predictions = torch.argmax(probs, dim=1).item()

  return "Ijobiy" if predictions == 1 else "Salbiy"

  # Bu predict_sentiment funksiyasi berilgan matnning ('text') sentimentini (ijobiy yoki salbiy) aniqlash uchun
  # ishlatiladi. U matnni tokenizatsiya qiladi, modeldan o'tkazib tahmin ehtimolliklarini oladi, eng yuqori
  # ehtimolga ega sinfni tanlaydi va natijani 'Ijobiy' yoki 'Salbiy' ko'rinishida qaytaradi.

In [ ]:

import gradio as gr

interface = gr.Interface(
    fn=predict_sentiment,
    inputs="text",
    outputs="text",
    title="Kino uchun izoh tahlili",
    description="Sizning fikiringiz o'rganilib kino uchun `ijobiy` yoki `salbiy` ekanligi aniqlanadi. `text` qismiga faqat inglizcha gap kiritish mumkin va uni natijasi uchun `Submit` qiling!"
)

interface.launch(share=True)

# Bu kod Gradio kutubxonasi yordamida kichik veb-interfeys yaratadi. predict_sentiment funksiyasini kirish sifatida
# 'text' olib, natijani 'text' ko'rinishida chiqaradi. title va description interfeysning sarlavhasi va tavsifini
# belgilaydi. interface.launch(share=True) esa bu veb-interfeyni ishga tushirib, uni umumiy foydalanish uchun havola yaratadi

# **MODELNI saqalash va load bilan qayta o'qish**

In [ ]:

torch.save(model.state_dict(), 'Sentiment_model.pth')
# Bu kod model.state_dict() yordamida modelning o'qitilgan parametrlarini (vazn va boshqa konfiguratsiyalarini)
# oladi va ularni 'model.pth' nomli faylga saqlaydi. Bu modelni keyinchalik qayta yuklash va ishlatish imkonini beradi.

In [ ]:
model.load_state_dict(torch.load('Sentiment_model.pth'))
model.eval()